In [1]:
# ========================================================================
# RANDOM FOREST MODEL: STEAM GAME REVENUE PREDICTION
# ========================================================================
# Comparing with XGBoost to evaluate model performance
# ========================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("=" * 70)
print("RANDOM FOREST MODEL: PREDICTING STEAM GAME REVENUE")
print("=" * 70)

RANDOM FOREST MODEL: PREDICTING STEAM GAME REVENUE


In [2]:
# ========================================================================
# 1. LOAD DATA AND CREATE REVENUE PROXY
# ========================================================================

print("\n" + "=" * 70)
print("STEP 1: LOADING DATA")
print("=" * 70)

df = pd.read_csv('steam_games_cleaned.csv')
print(f"\n✓ Dataset loaded: {df.shape[0]:,} games, {df.shape[1]} columns")

# Create revenue proxy
REVIEW_TO_OWNER_RATIO = 75
F2P_REVENUE_PER_PLAYER = 50

df['estimated_owners'] = df['overall_review_count'] * REVIEW_TO_OWNER_RATIO
df['estimated_revenue'] = df['estimated_owners'] * df['final_price']

df.loc[df['is_free_to_play'] == True, 'estimated_revenue'] = (
    df.loc[df['is_free_to_play'] == True, 'estimated_owners'] * F2P_REVENUE_PER_PLAYER
)

df['estimated_revenue'] = df['estimated_revenue'].fillna(0)

print(f"\n✓ Revenue proxy created")
print(f"Total estimated market: ₹{df['estimated_revenue'].sum()/1e9:.2f} Billion")


STEP 1: LOADING DATA

✓ Dataset loaded: 42,497 games, 35 columns

✓ Revenue proxy created
Total estimated market: ₹6004.71 Billion


In [3]:
# ========================================================================
# 2. FEATURE SELECTION & DATA PREPARATION
# ========================================================================

print("\n" + "=" * 70)
print("STEP 2: FEATURE SELECTION & DATA PREPARATION")
print("=" * 70)

FEATURES = [
    'main_genre',
    'final_price',
    'discount_pct_clean',
    'is_on_sale',
    'win_support',
    'mac_support',
    'linux_support',
    'multi_platform',
    'has_dlc',
    'dlc_available',
    'is_early_access',
    'is_free_to_play',
]

TARGET = 'estimated_revenue'

df_model = df[FEATURES + [TARGET]].copy()
df_model = df_model.dropna(subset=FEATURES)

zero_revenue_count = (df_model[TARGET] == 0).sum()
print(f"\n✓ Selected {len(FEATURES)} features")
print(f"✓ Dataset: {len(df_model):,} games (including {zero_revenue_count:,} zero-revenue)")


STEP 2: FEATURE SELECTION & DATA PREPARATION

✓ Selected 12 features
✓ Dataset: 4,844 games (including 527 zero-revenue)


In [4]:
# ========================================================================
# 3. FEATURE ENGINEERING
# ========================================================================

print("\n" + "=" * 70)
print("STEP 3: FEATURE ENGINEERING")
print("=" * 70)

# One-hot encode genre
genre_dummies = pd.get_dummies(df_model['main_genre'], prefix='genre', drop_first=False)
df_model = df_model.drop('main_genre', axis=1)
df_model = pd.concat([df_model, genre_dummies], axis=1)

# Convert boolean to int
bool_columns = ['win_support', 'mac_support', 'linux_support',
                'multi_platform', 'has_dlc', 'is_early_access',
                'is_free_to_play', 'is_on_sale']

for col in bool_columns:
    if col in df_model.columns:
        df_model[col] = df_model[col].astype(int)

print(f"✓ Feature engineering complete")
print(f"Final feature count: {len(df_model.columns) - 1}")


STEP 3: FEATURE ENGINEERING
✓ Feature engineering complete
Final feature count: 22


In [5]:
# ========================================================================
# 4. TRAIN-TEST SPLIT
# ========================================================================

print("\n" + "=" * 70)
print("STEP 4: TRAIN-TEST SPLIT")
print("=" * 70)

X = df_model.drop(TARGET, axis=1)
y = df_model[TARGET]
y_log = np.log1p(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

print(f"\n✓ Data split: {len(X_train):,} train, {len(X_test):,} test")

# Store genres for analysis
genre_cols = [col for col in X.columns if col.startswith('genre_')]

def get_genre_from_onehot(row, genre_cols):
    for col in genre_cols:
        if row[col] == 1:
            return col.replace('genre_', '')
    return 'unknown'

test_genres = X_test[genre_cols].apply(lambda row: get_genre_from_onehot(row, genre_cols), axis=1)


STEP 4: TRAIN-TEST SPLIT

✓ Data split: 3,875 train, 969 test


In [6]:
# ========================================================================
# 5. RANDOM FOREST - INITIAL MODEL
# ========================================================================

print("\n" + "=" * 70)
print("STEP 5: RANDOM FOREST - INITIAL MODEL")
print("=" * 70)

rf_initial = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

print("\n🌲 Training initial Random Forest model...")
rf_initial.fit(X_train, y_train)

y_pred_initial = rf_initial.predict(X_test)
initial_r2 = r2_score(y_test, y_pred_initial)
initial_rmse = np.sqrt(mean_squared_error(y_test, y_pred_initial))

print(f"\n✓ Initial model trained")
print(f"  Test R²: {initial_r2:.4f}")
print(f"  Test RMSE (log): {initial_rmse:.4f}")



STEP 5: RANDOM FOREST - INITIAL MODEL

🌲 Training initial Random Forest model...

✓ Initial model trained
  Test R²: 0.3191
  Test RMSE (log): 4.1397


In [7]:
# ========================================================================
# 6. GRID SEARCH HYPERPARAMETER TUNING
# ========================================================================

print("\n" + "=" * 70)
print("STEP 6: GRID SEARCH HYPERPARAMETER TUNING")
print("=" * 70)

print("\n🔍 Starting grid search (this may take 10-15 minutes)...")

param_grid = {
    'n_estimators': [200, 300, 400],
    'max_depth': [15, 20, 25],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

rf_base = RandomForestRegressor(random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    scoring='r2',
    cv=3,
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("\n✓ Grid search complete!")
print(f"\n📊 Best Parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV R² Score: {grid_search.best_score_:.4f}")


STEP 6: GRID SEARCH HYPERPARAMETER TUNING

🔍 Starting grid search (this may take 10-15 minutes)...
Fitting 3 folds for each of 162 candidates, totalling 486 fits

✓ Grid search complete!

📊 Best Parameters:
  max_depth: 25
  max_features: sqrt
  min_samples_leaf: 2
  min_samples_split: 10
  n_estimators: 200

Best CV R² Score: 0.2337


In [8]:
# ========================================================================
# 7. EVALUATE TUNED MODEL
# ========================================================================

print("\n" + "=" * 70)
print("STEP 7: EVALUATE TUNED RANDOM FOREST MODEL")
print("=" * 70)

rf_model = grid_search.best_estimator_

y_pred_train = rf_model.predict(X_train)
y_pred_test = rf_model.predict(X_test)

rf_r2_train = r2_score(y_train, y_pred_train)
rf_r2_test = r2_score(y_test, y_pred_test)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
rf_mae = mean_absolute_error(y_test, y_pred_test)

# Convert from log scale
y_test_original = np.expm1(y_test)
y_pred_test_original = np.expm1(y_pred_test)
rf_rmse_original = np.sqrt(mean_squared_error(y_test_original, y_pred_test_original))
rf_mae_original = mean_absolute_error(y_test_original, y_pred_test_original)

print(f"\n📊 Random Forest Performance:")
print(f"  Training R²: {rf_r2_train:.4f}")
print(f"  Test R²: {rf_r2_test:.4f}")
print(f"  Train-Test Gap: {rf_r2_train - rf_r2_test:.4f}")
print(f"\n  Test RMSE (log): {rf_rmse:.4f}")
print(f"  Test RMSE (₹): ₹{rf_rmse_original/1e6:.2f} Million")
print(f"  Test MAE (₹): ₹{rf_mae_original/1e6:.2f} Million")


STEP 7: EVALUATE TUNED RANDOM FOREST MODEL

📊 Random Forest Performance:
  Training R²: 0.4529
  Test R²: 0.3166
  Train-Test Gap: 0.1362

  Test RMSE (log): 4.1472
  Test RMSE (₹): ₹1005.60 Million
  Test MAE (₹): ₹104.06 Million


In [9]:
# ========================================================================
# 8. FEATURE IMPORTANCE ANALYSIS
# ========================================================================

print("\n" + "=" * 70)
print("STEP 8: FEATURE IMPORTANCE ANALYSIS")
print("=" * 70)

feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n📊 Top 15 Most Important Features:\n")
print(feature_importance.head(15).to_string(index=False))

# Category importance
genre_features = feature_importance[feature_importance['feature'].str.startswith('genre_')]
pricing_features = ['final_price', 'discount_pct_clean', 'is_on_sale']
pricing_importance = feature_importance[feature_importance['feature'].isin(pricing_features)]['importance'].sum()
genre_importance = genre_features['importance'].sum()

print("\n" + "-" * 70)
print("Feature Importance by Category:")
print("-" * 70)
print(f"  Genre features: {genre_importance:.4f} ({genre_importance*100:.1f}%)")
print(f"  Pricing & Discount: {pricing_importance:.4f} ({pricing_importance*100:.1f}%)")


STEP 8: FEATURE IMPORTANCE ANALYSIS

📊 Top 15 Most Important Features:

           feature  importance
       final_price    0.401034
discount_pct_clean    0.228917
     dlc_available    0.136345
           has_dlc    0.090543
       mac_support    0.025253
      genre_casual    0.024922
    multi_platform    0.018225
      genre_action    0.014403
     linux_support    0.013831
   is_early_access    0.011115
   genre_adventure    0.009937
       genre_indie    0.007622
         genre_rpg    0.006151
  genre_simulation    0.005093
    genre_strategy    0.004755

----------------------------------------------------------------------
Feature Importance by Category:
----------------------------------------------------------------------
  Genre features: 0.0747 (7.5%)
  Pricing & Discount: 0.6300 (63.0%)


In [10]:
# ========================================================================
# 9. GENRE-SPECIFIC RMSE ANALYSIS
# ========================================================================

print("\n" + "=" * 70)
print("STEP 9: GENRE-SPECIFIC RMSE ANALYSIS")
print("=" * 70)

genre_performance = pd.DataFrame({
    'genre': test_genres,
    'actual': y_test_original,
    'predicted': y_pred_test_original,
    'squared_error': (y_test_original - y_pred_test_original)**2
})

genre_rmse = genre_performance.groupby('genre').agg({
    'squared_error': lambda x: np.sqrt(x.mean()),
    'actual': ['count', 'mean', 'std']
}).round(2)

genre_rmse.columns = ['RMSE', 'Sample_Count', 'Mean_Revenue', 'Std_Revenue']
genre_rmse['RMSE_millions'] = (genre_rmse['RMSE'] / 1e6).round(2)
genre_rmse['Mean_Revenue_M'] = (genre_rmse['Mean_Revenue'] / 1e6).round(2)

genre_rmse_sorted = genre_rmse.sort_values('RMSE', ascending=False)

print("\nTop 10 Genres by Prediction Difficulty (RMSE):\n")
print(genre_rmse_sorted[['Sample_Count', 'RMSE_millions', 'Mean_Revenue_M']].head(10).to_string())


STEP 9: GENRE-SPECIFIC RMSE ANALYSIS

Top 10 Genres by Prediction Difficulty (RMSE):

            Sample_Count  RMSE_millions  Mean_Revenue_M
genre                                                  
rpg                   19        5213.07         1461.40
action               434        1017.97          151.29
strategy              15         849.49          264.25
indie                 94         147.84           31.46
sports                 1         100.20          127.58
adventure            206          64.54           14.84
racing                 5          50.48           15.17
simulation            23          14.98            6.85
casual               172          13.27            2.40


In [11]:
# ========================================================================
# 10. VISUALIZATIONS
# ========================================================================

print("\n" + "=" * 70)
print("STEP 10: CREATING VISUALIZATIONS")
print("=" * 70)

sns.set(style="whitegrid", palette="muted")

# Visualization 1: Feature Importance
fig, ax = plt.subplots(figsize=(10, 8))
top_features = feature_importance.head(15).sort_values('importance')
ax.barh(top_features['feature'], top_features['importance'], color='forestgreen')
ax.set_xlabel('Importance', fontsize=12, fontweight='bold')
ax.set_ylabel('Feature', fontsize=12, fontweight='bold')
ax.set_title('Top 15 Feature Importance (Random Forest)', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=300, bbox_inches='tight')
print("✓ Saved: rf_feature_importance.png")
plt.close()

# Visualization 2: Actual vs Predicted
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(y_test_original/1e6, y_pred_test_original/1e6, alpha=0.3, s=20, color='forestgreen')
max_val = max(y_test_original.max(), y_pred_test_original.max()) / 1e6
ax.plot([0, max_val], [0, max_val], 'r--', lw=2, label='Perfect Prediction')
ax.set_xlabel('Actual Revenue (₹ Millions)', fontsize=12, fontweight='bold')
ax.set_ylabel('Predicted Revenue (₹ Millions)', fontsize=12, fontweight='bold')
ax.set_title(f'Random Forest: Actual vs Predicted (R² = {rf_r2_test:.3f})',
             fontsize=14, fontweight='bold', pad=20)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('rf_actual_vs_predicted.png', dpi=300, bbox_inches='tight')
print("✓ Saved: rf_actual_vs_predicted.png")
plt.close()


STEP 10: CREATING VISUALIZATIONS
✓ Saved: rf_feature_importance.png
✓ Saved: rf_actual_vs_predicted.png


In [12]:
# ========================================================================
# 11. SAVE RESULTS
# ========================================================================

print("\n" + "=" * 70)
print("STEP 11: SAVING RESULTS")
print("=" * 70)

# Save feature importance
feature_importance.to_csv('rf_feature_importance.csv', index=False)
print("✓ Saved: rf_feature_importance.csv")

# Save predictions
predictions_df = pd.DataFrame({
    'actual_revenue': y_test_original,
    'predicted_revenue': y_pred_test_original,
    'error': y_test_original - y_pred_test_original,
    'genre': test_genres.values
})
predictions_df.to_csv('rf_predictions.csv', index=False)
print("✓ Saved: rf_predictions.csv")

# Save genre RMSE
genre_rmse_sorted.to_csv('rf_genre_rmse_analysis.csv')
print("✓ Saved: rf_genre_rmse_analysis.csv")

# Save model
import joblib
joblib.dump(rf_model, 'rf_revenue_model.pkl')
print("✓ Saved: rf_revenue_model.pkl")

# Save best parameters
import json
with open('rf_best_hyperparameters.json', 'w') as f:
    json.dump(grid_search.best_params_, f, indent=4)
print("✓ Saved: rf_best_hyperparameters.json")

# Save summary
summary = {
    'Metric': ['Model', 'Training Samples', 'Test Samples', 'Features',
               'Train R²', 'Test R²', 'Train-Test Gap',
               'Test RMSE (₹M)', 'Test MAE (₹M)'],
    'Value': ['Random Forest', len(X_train), len(X_test), X_train.shape[1],
              f'{rf_r2_train:.4f}', f'{rf_r2_test:.4f}', f'{rf_r2_train - rf_r2_test:.4f}',
              f'{rf_rmse_original/1e6:.2f}', f'{rf_mae_original/1e6:.2f}']
}
summary_df = pd.DataFrame(summary)
summary_df.to_csv('rf_model_summary.csv', index=False)
print("✓ Saved: rf_model_summary.csv")

print("\n" + "=" * 70)
print("✓ RANDOM FOREST MODEL TRAINING COMPLETE!")
print("=" * 70)

print("\n📊 FINAL RESULTS:")
print(f"   Test R²: {rf_r2_test:.4f}")
print(f"   Test RMSE: ₹{rf_rmse_original/1e6:.2f}M")
print(f"   Train-Test Gap: {rf_r2_train - rf_r2_test:.4f}")


STEP 11: SAVING RESULTS
✓ Saved: rf_feature_importance.csv
✓ Saved: rf_predictions.csv
✓ Saved: rf_genre_rmse_analysis.csv
✓ Saved: rf_revenue_model.pkl
✓ Saved: rf_best_hyperparameters.json
✓ Saved: rf_model_summary.csv

✓ RANDOM FOREST MODEL TRAINING COMPLETE!

📊 FINAL RESULTS:
   Test R²: 0.3166
   Test RMSE: ₹1005.60M
   Train-Test Gap: 0.1362


In [13]:
# ========================================================================
# VERIFY ZERO REVENUE GAMES ARE INCLUDED
# ========================================================================

print("\n" + "=" * 70)
print("VERIFICATION: ZERO REVENUE GAMES IN TRAINING")
print("=" * 70)

# Check target variable
y_original = np.expm1(y_log)  # Convert back from log
zero_count = (y_original == 0).sum()
total_count = len(y_original)

print(f"\nTarget variable (y_log) after log1p transformation:")
print(f"  Total samples: {total_count:,}")
print(f"  Zero revenue games: {zero_count:,} ({zero_count/total_count*100:.2f}%)")
print(f"  Non-zero revenue games: {total_count - zero_count:,} ({(total_count - zero_count)/total_count*100:.2f}%)")

# Check if model can predict near-zero
print(f"\nModel prediction range:")
print(f"  Minimum prediction: ₹{np.expm1(y_pred_test).min()/1e6:.2f}M")
print(f"  Maximum prediction: ₹{np.expm1(y_pred_test).max()/1e6:.2f}M")
print(f"  Predictions < ₹1M: {(np.expm1(y_pred_test) < 1e6).sum():,}")

if (y_original == 0).sum() > 0:
    print("\n✅ CONFIRMED: Zero revenue games are included in training")
else:
    print("\n⚠️ WARNING: No zero revenue games found (they may have been filtered)")


VERIFICATION: ZERO REVENUE GAMES IN TRAINING

Target variable (y_log) after log1p transformation:
  Total samples: 4,844
  Zero revenue games: 527 (10.88%)
  Non-zero revenue games: 4,317 (89.12%)

Model prediction range:
  Minimum prediction: ₹0.00M
  Maximum prediction: ₹279.48M
  Predictions < ₹1M: 717

✅ CONFIRMED: Zero revenue games are included in training
